# CS383: Data Science and Machine Learning
## Lecture 8 — Your First Classifiers: Logistic Regression & k-NN

*Dr. Thitima Srivatanakul*

### Guiding question
**Last week you fit a line to predict a number. This week: what changes when the question isn't
"how much" but "which one"?**

### Learning objectives
By the end of this lecture, you should be able to:

- explain the difference between regression (predicting a number) and classification (predicting a
  class)
- explain why ordinary linear regression is inappropriate for predicting a binary outcome
- explain, at a high level, how logistic regression uses a sigmoid function to turn a linear score into
  a probability
- distinguish a predicted **probability**, a **threshold** (often 0.5), and a **predicted class**
- fit and use logistic regression with scikit-learn (`.fit()`, `.predict()`, `.predict_proba()`)
- explain k-NN as classification by looking at nearby examples and voting
- explain how the choice of $k$ affects how flexible a k-NN boundary is
- explain why feature scaling matters for k-NN
- evaluate a classifier's accuracy on test data and compare it against a simple majority-class baseline

*(Curious how logistic regression's coefficients might be found, conceptually? The "Optional /
Under the Hood" section partway through Part 1 works through log-loss and one simple version of
gradient descent in full -- worth a look, but not required.)*

---

### Where this fits
Lecture 7 fit a regression line, then tested it against real 311 and restaurant data. Today follows the
same arc for classification: Part 1 reuses last lecture's twelve-student toy set to build logistic
regression -- from why a straight line fails, through the sigmoid, to a fitted probability curve --
while keeping the mechanics of *how* `.fit()` finds the coefficients optional. Part 2 introduces a
second toy dataset -- 40 restaurants awaiting reinspection -- to compare logistic regression's
straight-line boundary against k-NN's locally-adaptive one. Part 3 checks both against the real
restaurant inspections data from Lectures 6 and 7, and previews the sharper evaluation tools coming in
Lecture 9.

---

### Before we open the notebook: an unplugged warm-up

No coding for this part. You'll sort a small set of students into "pass" or "fail" by eye, meet the
sigmoid curve that turns a straight line into a probability, and compare how logistic regression and
k-NN each draw the boundary between two classes — one by fitting a formula, the other by comparing
distances with no training step at all.

**[Open the "Meet Your First Classifiers" Activity](https://thitimas.github.io/cs383-fa26-materials-public/lect08/classification_unplugged_activity.html)**

Takes about 15-20 minutes. Come back here once you've been through all the rounds.

---

## Part 1 — Logistic Regression, From Scratch

Same twelve students, same hours studied, same exam scores as Lecture 7 — but a different question this
time. Instead of "what score will they get," let's ask: **did they pass?**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression

HOURS = np.array([1, 2, 3, 3, 4, 5, 6, 6, 7, 8, 9, 10], dtype=float)
SCORES = np.array([50, 54, 60, 58, 65, 64, 72, 75, 78, 82, 88, 90], dtype=float)

# "Pass" = a score of 65 or better. Notice this is *not* a clean cutoff on hours studied: the
# 4-hour student scored 65 (a pass) and the 5-hour student scored 64 (a near-miss fail) -- hours
# studied matters a lot, but it isn't the only thing that decides an exam score.
PASS = (SCORES >= __________).astype(int)

for h, s, p in zip(HOURS, SCORES, PASS):
    print(f"{h:.0f} hours -> score {s:.0f} -> {'PASS' if p else 'fail'}")

In [ ]:
rng_jitter = np.random.default_rng(383)
y_jittered = PASS + rng_jitter.uniform(-0.03, 0.03, size=len(PASS))

plt.scatter(HOURS, y_jittered, c=PASS, cmap="coolwarm", s=90, edgecolor="white", linewidth=1)
plt.yticks([0, 1], ["fail", "pass"])
plt.xlabel("Hours studied")
plt.ylabel("Outcome")
plt.title("Pass/Fail vs. Hours Studied")
plt.show()

The outcome only ever takes two values, 0 or 1 -- there's no "in-between" exam result. A straight line,
fit the way Lecture 7 fit one, doesn't know that. Let's actually try it and see what goes wrong.

In [ ]:
naive_model = __________()
naive_model.fit(HOURS.reshape(-1, 1), PASS)

print(f"slope:     {naive_model.coef_[0]:.4f}")
print(f"intercept: {naive_model.intercept_:.4f}")
print()
for h in [0, 1, 10, 12, 15]:
    pred = naive_model.predict([[h]])[0]
    print(f"hours = {h:>2}: predicted \"probability\" of passing = {pred:.3f}")

At 0 hours studied, this "model" predicts a probability of about **-0.18** -- negative, which isn't a
valid probability at all. At 15 hours, it predicts about **1.96** -- also not valid; probabilities can
only live between 0 and 1. A straight line has no idea it's supposed to stay in that range, because
nothing in its definition constrains it to. We need a function that takes any real number in, and always
hands back something between 0 and 1.

### The sigmoid function

That function is the **sigmoid** (or logistic) function:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Whatever real number $z$ you feed it -- huge and negative, huge and positive, anything in between --
$\sigma(z)$ always comes back squeezed into $(0, 1)$. Large negative $z$ pushes the output toward 0;
large positive $z$ pushes it toward 1; $z=0$ lands exactly at 0.5.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.__________(-z))

z_range = np.linspace(-8, 8, 200)
plt.plot(z_range, sigmoid(z_range), color="#2B6CB0", linewidth=2.5)
plt.axhline(0.5, color="#9AA5B1", linestyle="--", linewidth=1)
plt.axvline(0, color="#9AA5B1", linestyle="--", linewidth=1)
plt.xlabel("z")
plt.ylabel("sigmoid(z)")
plt.title("The Sigmoid Function")
plt.show()

### Putting it together: logistic regression

Logistic regression starts exactly like linear regression -- the same weighted sum of features:

$$z = b_0 + b_1 x$$

but then squashes that number through the sigmoid before calling it a prediction:

$$p = \sigma(z) = \sigma(b_0 + b_1 x)$$

$p$ is the model's predicted **probability** that the outcome is 1 (pass). Same linear combination of
features as before, same $b_0$/$b_1$ coefficients to find -- just with one extra step that keeps the
output honest as a probability.

### Fitting it with scikit-learn

Just like linear regression last week, we don't have to find $b_0$ and $b_1$ by hand -- call `.fit()`
and let scikit-learn find them. (*How* it finds them is worth understanding conceptually, and we'll get
there in a moment -- but let's see the result first.)

In [ ]:
model = LogisticRegression()
model.__________(HOURS.reshape(-1, 1), PASS)

print(f"intercept (b0): {model.intercept_[0]:.4f}")
print(f"slope     (b1): {model.coef_[0][0]:.4f}")

In [ ]:
hours_range = np.linspace(0, 11, 200)
fitted_curve = sigmoid(model.intercept_[0] + model.__________[0][0] * hours_range)

plt.scatter(HOURS, y_jittered, c=PASS, cmap="coolwarm", s=90, edgecolor="white", linewidth=1, zorder=3)
plt.plot(hours_range, fitted_curve, color="#2B6CB0", linewidth=2.5, label="fitted sigmoid")
plt.axhline(0.5, color="#9AA5B1", linestyle="--", linewidth=1)
plt.yticks([0, 0.5, 1], ["fail (0)", "0.5", "pass (1)"])
plt.xlabel("Hours studied")
plt.ylabel("Predicted probability of passing")
plt.title("Fitted Logistic Regression Curve")
plt.legend()
plt.show()

This S-curve *is* the model -- everything else is detail. Low hours studied puts you on the flat left
tail, where predicted probability sits near 0. High hours puts you on the flat right tail, near 1. In
between, the curve climbs fast: around 4-5 hours, where the toy data itself is genuinely ambiguous (4
hours passed; 5 hours barely failed), the curve crosses 0.5 -- the model is honestly uncertain right
where the data is.

In [ ]:
sample_hours = np.array([[3], [5.5], [9]])

probabilities = model.predict_proba(sample_hours)
predictions = model.__________(sample_hours)

for h, proba, pred in zip(sample_hours.ravel(), probabilities, predictions):
    print(f"{h} hours -> P(fail)={proba[0]:.3f}, P(pass)={proba[1]:.3f} -> predict: {'pass' if pred else 'fail'}")

Three different things are happening in that one line of output, and it's worth keeping them separate:

- **`predict_proba()` -> probability.** The model's actual estimate -- e.g. "82.4% chance of passing."
  This is the curve you just plotted, evaluated at one point.
- **Threshold -> decision rule.** Somewhere you have to draw a line and call it a decision. By default,
  `.predict()` uses **0.5**: probability above 0.5 becomes "pass," at or below becomes "fail." 0.5 is a
  default, not a law of nature -- Lecture 9 revisits when a different threshold makes more sense.
- **`predict()` -> predicted class.** Just the threshold applied to the probability, returned as a plain
  label.

Most of the time you'll reach for `.predict()`. Reach for `.predict_proba()` whenever "how confident was
it?" matters more than a bare yes/no.

### Why fitting needs a loss function

`.fit()` didn't guess $b_0$ and $b_1$ -- it searched for the values that make the model's probabilities
match the data as closely as possible. To search, it needs a way to score "how wrong" a set of
coefficients is. That score is called **log-loss**, and the idea behind it is simple:

- correct **and** confident -> small loss
- wrong **and** confident -> large loss
- lower loss -> better fit

(For reference, the formula is $L = -\frac{1}{n}\sum_{i=1}^n \big[y_i\log(p_i) + (1-y_i)\log(1-p_i)\big]$
-- you don't need to memorize or derive it. The intuition above is what matters.)

### Why fitting needs gradient descent

Linear regression's best line had a closed-form formula -- one algebra step, done. Logistic regression's
log-loss doesn't have that shortcut, so its coefficients are found using an **iterative optimization
algorithm** instead. **Gradient descent** is one simple way to understand the general idea:

1. start with some coefficients (even just 0)
2. use them to predict probabilities
3. measure the loss
4. adjust the coefficients a little, in whichever direction reduces the loss
5. repeat, thousands of times, until the loss stops improving

In practice, scikit-learn handles the optimization for us when we call `.fit()` -- it doesn't
necessarily run this exact procedure internally, but the idea above -- predict, measure loss, adjust,
repeat -- is the core of what's happening. The "Optional / Under the Hood" section right below codes a
simple version of gradient descent by hand, if you want to see one concrete way this kind of
optimization can work; skip it if the conceptual version above is enough for now.

---

### *Optional / Under the Hood:* Log-loss and gradient descent, worked by hand

Everything above is what you need for the rest of this course. This section is for anyone curious about
one concrete way this kind of optimization can work -- coding a simple version of gradient descent by
hand, watching it converge, and comparing the result to scikit-learn's own answer (found using its own
optimization routine, not necessarily this exact procedure).

**Log-loss, in full:**

$$L = -\frac{1}{n}\sum_{i=1}^{n} \Big[ y_i \log(p_i) + (1 - y_i)\log(1 - p_i) \Big]$$

Look at what each term does. When the true label $y_i = 1$: only the $\log(p_i)$ term survives. If the
model predicted $p_i$ close to 1 (confident and right), $\log(p_i)$ is close to 0 -- almost no loss. If
it predicted $p_i$ close to 0 (confident and *wrong*), $\log(p_i)$ shoots toward $-\infty$ -- a massive
penalty. Same logic in reverse when $y_i = 0$. That's the mechanism behind "confidently wrong is
punished far more severely than being merely a little off.\"

In [ ]:
def log_loss(y, p, eps=1e-9):
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

b0, b1 = 0.0, 0.0
learning_rate = 0.05

for iteration in range(20_000):
    z = b0 + b1 * HOURS
    p = sigmoid(z)
    grad_b0 = np.mean(p - PASS)
    grad_b1 = np.mean((p - PASS) * HOURS)
    b0 -= learning_rate * __________
    b1 -= learning_rate * grad_b1
    if iteration in (0, 100, 1000, 5000, 19999):
        print(f"iteration {iteration:>6}: loss = {log_loss(PASS, p):.4f}, b0 = {b0:.3f}, b1 = {b1:.3f}")

print()
print(f"By-hand gradient descent: b0 = {b0:.4f}, b1 = {b1:.4f}")

Watch the loss column: it keeps dropping, iteration after iteration, but there's no single algebraic
step where it "solves" for the answer the way Lecture 7's `b1 = Sxy / Sxx` did. It just keeps taking
small steps downhill on the bowl-shaped loss surface, the same kind of bowl you saw for RSS -- except
this bowl has no closed-form bottom to jump to directly, so gradient descent has to walk there.

*Reference figure — for visual intuition, not something you need to memorize.*

![A smooth bowl-shaped 3D loss surface, shown empty and with a ball resting at the bottom](images/bowl-shaped-loss-surface.png)

This is the "bowl" the explanation above is describing: a smooth surface with one lowest point. Gradient descent's job is exactly what's pictured on the right -- start somewhere on the slope and keep taking small downhill steps (following the gradient) until the ball settles at the bottom, which is the combination of $b_0$ and $b_1$ that minimizes the loss.

*Figure adapted from Andrew Glassner's* Deep Learning: A Visual Approach *(No Starch Press), released
under the MIT license.*

In [ ]:
# penalty=None turns off scikit-learn's default regularization, so this is an apples-to-apples
# comparison with our own unregularized gradient descent above. (By default, LogisticRegression()
# regularizes automatically -- generally what you want in practice, and a preview of Ridge/Lasso
# later in the course.)
sklearn_model = LogisticRegression(penalty=None, max_iter=5000)
sklearn_model.__________(HOURS.reshape(-1, 1), PASS)

print(f"scikit-learn: b0 = {sklearn_model.intercept_[0]:.4f}, b1 = {sklearn_model.coef_[0][0]:.4f}")
print(f"by hand:      b0 = {b0:.4f}, b1 = {b1:.4f}")

Close, not identical -- and that gap *is* the point. Linear regression's by-hand answer matched
scikit-learn's to the decimal, because both were solving the same algebra problem. Here, both are
searching the same bowl-shaped loss surface for its lowest point -- our hand-coded version by walking
downhill step by step (gradient descent), scikit-learn's own fit by its own optimization routine -- and
gradient descent only ever gets *close* to the bottom; how close depends on the learning rate and how
many steps you let it take. That's the tradeoff every model without a closed-form solution has to live
with -- including logistic regression, and later, neural networks.

---

## Part 2 — Decision Boundaries: Two Very Different Ways to Classify

A toy dataset for this part: imagine 40 restaurants that failed an inspection and were scheduled for a
reinspection. Two features -- how many violations they racked up, and how many days they had before the
reinspection -- and one label: did they pass? *(Illustrative numbers built to teach the mechanics
cleanly, not a real dataset -- Part 3 checks these same ideas against actual NYC inspection data.)*

In [ ]:
VIOLATIONS = np.array([2, 0, 9, 8, 2, 9, 2, 8, 5, 3, 5, 0, 5, 2, 5, 3, 2, 5, 2, 5,
                       7, 0, 5, 6, 0, 6, 6, 5, 8, 10, 4, 10, 9, 7, 0, 10, 4, 0, 8, 6], dtype=float)
DAYS_UNTIL_REINSPECTION = np.array([258, 228, 273, 47, 323, 302, 322, 24, 256, 37, 24, 218, 24, 8, 341,
                                     114, 184, 229, 99, 310, 6, 351, 296, 374, 377, 137, 365, 104, 70,
                                     259, 155, 324, 225, 321, 167, 128, 298, 344, 347, 340], dtype=float)
PASSES_REINSPECTION = np.array([1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0,
                                 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0])

X_toy = np.__________([VIOLATIONS, DAYS_UNTIL_REINSPECTION])
print(f"{len(PASSES_REINSPECTION)} restaurants, {PASSES_REINSPECTION.sum()} passed reinspection")

The helper below fits a classifier, then colors every point in the feature space by what that
classifier would predict there -- that colored background *is* the decision boundary made visible.

In [ ]:
def plot_decision_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 20, X[:, 1].max() + 20
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))

    grid_preds = model.predict(np.column_stack([xx.ravel(), yy.ravel()]))
    grid_preds = grid_preds.reshape(xx.shape)

    plt.contourf(xx, yy, grid_preds, levels=[-0.5, 0.5, 1.5], colors=["#F7D9D6", "#D6E4F0"], alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=70, edgecolor="white", linewidth=1)
    plt.xlabel("Violations")
    plt.ylabel("Days until reinspection")
    plt.title(title)
    plt.show()

In [ ]:
logreg_toy = LogisticRegression()
logreg_toy.__________(X_toy, PASSES_REINSPECTION)

plot_decision_boundary(logreg_toy, X_toy, PASSES_REINSPECTION, "Logistic Regression Decision Boundary")

One straight edge splits the whole plane in two. That's not a limitation of this particular fit --
logistic regression's decision boundary (where $p = 0.5$) is *always* a straight line (or a flat plane,
with more features), because $z = b_0 + b_1x_1 + b_2x_2$ is linear no matter how you slice it. If the
true boundary between classes actually curves, logistic regression can't bend to follow it.

*Reference figure — for visual intuition, not something you need to memorize.*

![Two classes of points separated by a single straight decision-boundary line](images/linear-decision-boundary.png)

This is what "one straight edge splits the whole plane in two" looks like in general: whatever the two features are, logistic regression can only ever carve the feature space with a straight line (or, with more features, a flat plane). It can rotate and shift that line to fit the data as well as possible, but it can never bend it.

*Figure adapted from Andrew Glassner's* Deep Learning: A Visual Approach *(No Starch Press), released
under the MIT license.*

### k-Nearest Neighbors (k-NN)

k-NN throws out the idea of fitting a formula entirely. To classify a new point, it:

1. measures the distance from that point to every training point,
2. finds the $k$ closest ones,
3. and predicts whichever class is the majority among those $k$ neighbors.

There's no `.fit()` doing any real work here -- "training" a k-NN model just means storing the training
data. All the computation happens at prediction time, one neighborhood at a time.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_1 = KNeighborsClassifier(n_neighbors=__________)
knn_1.fit(X_toy, PASSES_REINSPECTION)

plot_decision_boundary(knn_1, X_toy, PASSES_REINSPECTION, "k-NN Decision Boundary (k=1)")

With $k=1$, every prediction is decided by the single closest training point -- so the boundary snakes
tightly around each point's little island of influence, chasing every quirk in the training data
(including whatever's just noise). It gets 100% accuracy *on the training data itself*, which should
worry you, not impress you: a model that perfectly memorizes its training set usually isn't learning the
underlying pattern, just the specific 40 points it was shown.

In [ ]:
knn_15 = KNeighborsClassifier(n_neighbors=15)
knn_15.__________(X_toy, PASSES_REINSPECTION)

plot_decision_boundary(knn_15, X_toy, PASSES_REINSPECTION, "k-NN Decision Boundary (k=15)")

With $k=15$, each prediction is an average over more neighbors, so one noisy point nearby can't flip
the answer on its own -- the boundary smooths out. Push $k$ too far in the other direction (say, $k=$ the
whole dataset) and it stops looking at neighbors at all: every prediction just becomes the overall
majority class. Choosing $k$ is a genuine tradeoff between chasing individual points too closely and
ignoring local structure altogether -- you'll formalize that tradeoff properly, under the name
*bias-variance*, later in the course.

*Reference figure — for visual intuition, not something you need to memorize.*

![The same two classes of points, now separated by a flexible, wiggly decision boundary that threads between them](images/flexible-decision-boundary.png)

Contrast this with the straight-line boundary above. A flexible, instance-based boundary like this can bend and wiggle to fit the training points closely -- which is the same trade-off you just saw between $k=1$ and $k=15$: more flexibility can mean a tighter fit to training data, but also more risk of chasing noise.

*Figure adapted from Andrew Glassner's* Deep Learning: A Visual Approach *(No Starch Press), released
under the MIT license.*

### Two different ways to classify

Side by side: logistic regression fits a formula by minimizing a loss; k-NN skips fitting entirely and
just compares a new point to its nearest stored neighbors.

| | Logistic regression | k-NN |
|---|---|---|
| "Training" | fits $b_0, b_1, \dots$ by minimizing a loss | just stores the data |
| Decision boundary shape | always linear | can be any shape, adapts locally |
| Prediction cost | one formula evaluation | compares to every training point |

(These two styles are sometimes called *parametric* vs. *instance-based* -- useful labels to recognize
if you see them again, but not something to memorize hard right now.)

### Why k-NN needs scaled features

k-NN's whole judgment of "closest" comes from distance -- so if one feature's raw range dwarfs another's,
that feature silently dominates every distance calculation, exactly like Lecture 6's age-vs-income
example. Here, `days_until_reinspection` ranges over hundreds while `violations` only ranges 0-10. Let's
see it happen.

In [ ]:
target = X_toy[0]           # (2 violations, 258 days)
candidate_a = X_toy[1]      # (0 violations, 228 days) -- close on violations, off on days
candidate_b = X_toy[2]      # (9 violations, 273 days) -- way off on violations, close on days

dist_a_unscaled = np.linalg.norm(candidate_a - target)
dist_b_unscaled = np.linalg.__________(candidate_b - target)

print(f"target:      {target}")
print(f"candidate A: {candidate_a}  (unscaled distance: {dist_a_unscaled:.2f})")
print(f"candidate B: {candidate_b}  (unscaled distance: {dist_b_unscaled:.2f})")
print()
print("Unscaled, k-NN would call", "A" if dist_a_unscaled < dist_b_unscaled else "B", "the closer neighbor.")

Candidate B differs from the target by **7 more violations** -- a huge gap on a 0-10 scale -- but is
called the closer neighbor anyway, purely because its 15-day gap on `days_until_reinspection` is smaller
than candidate A's 30-day gap. A 7-violation difference getting outweighed by a 15-day difference is
backwards from what most people would call "similar." Now scale both features and check again.

In [ ]:
from sklearn.preprocessing import StandardScaler

X_toy_scaled = StandardScaler().__________(X_toy)

target_s = X_toy_scaled[0]
dist_a_scaled = np.linalg.norm(X_toy_scaled[1] - target_s)
dist_b_scaled = np.linalg.norm(X_toy_scaled[2] - target_s)

print(f"scaled distance to A: {dist_a_scaled:.2f}")
print(f"scaled distance to B: {dist_b_scaled:.2f}")
print()
print("Scaled, k-NN now calls", "A" if dist_a_scaled < dist_b_scaled else "B", "the closer neighbor.")

The ranking flips. Once both features are standardized to the same footing, candidate A -- the one that
actually matches more closely on violations -- wins out, the way it should have all along. **k-NN is
sensitive to feature scale, so numeric features are usually scaled before using it.** Logistic
regression's coefficients can adjust to a feature's raw scale on their own; k-NN's raw distance
calculation cannot.

---

## Part 3 — Does This Work on Real Data?

Same restaurant inspections dataset as Lectures 6 and 7. This time, `is_critical` -- whether a violation
citation was flagged critical -- becomes the *target* instead of a feature, and `score` becomes a
predictor instead of the thing being predicted.

### Setup — NYC restaurant inspections

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

try:
    raw_path = os.path.expanduser("~/shared/restaurant_inspections_snapshot.csv")
    inspections_df = pd.read_csv(raw_path)
    inspections_df["score"] = pd.to_numeric(inspections_df["score"], errors="coerce")
    inspections_df = inspections_df.dropna(subset=["score", "boro", "cuisine_description"]).reset_index(drop=True)
    inspections_df["is_critical"] = (inspections_df["critical_flag"] == "Critical").astype(int)

    top_cuisines = inspections_df["cuisine_description"].value_counts().nlargest(10).index
    inspections_df["cuisine_top"] = np.where(
        inspections_df["cuisine_description"].isin(top_cuisines), inspections_df["cuisine_description"], "Other"
    )
    live = True

except Exception:
    rng = np.random.default_rng(383)
    n = 4000
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza", "Japanese"]

    inspections_df = pd.DataFrame({
        "boro": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_top": rng.choice(cuisines_clean, size=n),
        "score": rng.integers(0, 71, size=n),
    })
    inspections_df["is_critical"] = rng.integers(0, 2, size=n)
    live = False

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(inspections_df):,} violation records")
print(f"is_critical balance: {inspections_df['is_critical'].mean():.3f}")

### The baseline you have to beat

Before fitting anything, ask: what's the accuracy of the dumbest possible classifier -- always guessing
the majority class? If `is_critical` were 90% "yes," a model that always predicts "yes" would already be
90% accurate without learning a thing. Any real model needs to clear that bar, not just clear 50%.

In [ ]:
X_single = inspections_df[["score"]]
y = inspections_df["is_critical"]

X_train, X_test, y_train, y_test = train_test_split(X_single, y, test_size=__________, random_state=383, stratify=y)

majority_class = y_train.mode()[0]
majority_baseline = (y_test == majority_class).mean()
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")

In [ ]:
single_model = LogisticRegression()
single_model.__________(X_train, y_train)

y_pred = single_model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}  (baseline: {majority_baseline:.3f})")
print(f"Coefficient on score: {single_model.coef_[0][0]:.4f}")

Around 59% accuracy against a 56% baseline -- real, but modest. The positive coefficient on `score`
makes domain sense (a worse inspection score is associated with a somewhat higher chance of a critical
violation). That's what most real classification problems actually look like: genuine signal, but far
from a clean separation.

Accuracy is a reasonable first check, but it's not the whole story -- Lecture 9 introduces sharper tools
(like AUC and confusion matrices) for judging a classifier more carefully, especially when, like here,
one class is more common than the other.

In [ ]:
X_multi = inspections_df[["score", "boro", "cuisine_top"]]

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y, test_size=0.2, random_state=383, stratify=y
)

preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["boro", "cuisine_top"]),
], remainder="passthrough")

X_train_ready = preprocessor.fit_transform(X_train_m)
X_test_ready = preprocessor.__________(X_test_m)

multi_model = LogisticRegression(max_iter=1000)
multi_model.fit(X_train_ready, y_train_m)

y_pred_m = multi_model.predict(X_test_ready)

print(f"Accuracy: {accuracy_score(y_test_m, y_pred_m):.3f}  (single-feature was: {accuracy_score(y_test, y_pred):.3f})")

Adding `boro` and `cuisine_top` nudges accuracy up only a little. Two more features bought almost
nothing here -- a completely normal, honest result, not a mistake in the code.

k-NN works on this same real data exactly the way it did on the toy dataset in Part 2 -- fit a
`KNeighborsClassifier` the same way, remembering to scale `score` first. The exercise notebook has you
do exactly that, on a different real dataset.

---

**Exercises for this lecture** (Lab, Exit Ticket, Optional Challenge) live in a separate notebook:
`lect08_classification_intro_exercise.ipynb`.

---

## Cheat Sheet

| Task | Code |
|---|---|
| Fit a logistic regression | `LogisticRegression().fit(X_train, y_train)` |
| Predict class labels | `model.predict(X_test)` |
| Predict class probabilities | `model.predict_proba(X_test)` |
| Fit a k-NN classifier | `KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)` |
| Scale features before k-NN | `StandardScaler().fit_transform(X_train)` |
| Accuracy | `accuracy_score(y_test, y_pred)` |
| Majority-class baseline | `max(y.mean(), 1 - y.mean())` |

---

## Key Terms

- **Classification**: predicting a category (a class label) instead of a continuous number.
- **Sigmoid function**: $\sigma(z) = 1/(1+e^{-z})$; squashes any real number into $(0, 1)$.
- **Logistic regression**: a linear combination of features, passed through the sigmoid, trained by
  minimizing log-loss.
- **Probability, threshold, predicted class**: `predict_proba()` gives the model's estimated
  probability; a threshold (0.5 by default) turns that probability into a decision rule; `predict()`
  applies the threshold and returns the resulting class.
- **Log-loss (cross-entropy)**: the loss function logistic regression minimizes; conceptually,
  correct-and-confident predictions score low, wrong-and-confident predictions score high.
- **Gradient descent**: an iterative recipe -- predict, measure loss, adjust coefficients, repeat --
  one simple way to understand how methods without a closed-form formula find their minimum.
  `.fit()` handles the actual optimization for us, not necessarily via this exact procedure.
- **Decision boundary**: the line (or curve) in feature space separating where a classifier predicts one
  class from where it predicts another.
- **k-Nearest Neighbors (k-NN)**: an instance-based classifier that predicts the majority class among a
  new point's $k$ closest training points; has no training step beyond storing the data.
- **Majority-class baseline**: the accuracy you'd get by always predicting the more common class; the
  bar any real classifier needs to clear.